In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import WebBaseLoader
from langchain.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser

In [11]:
from IPython.display import Markdown
def to_Markdown(text):
    return Markdown(text)

In [12]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
def VectorStore(data,embedding):
    splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap =500)
    chunks = splitter.split_documents(data)
    vector = FAISS.from_documents(chunks,embedding)
    retriever = vector.as_retriever()
    return retriever



In [13]:
llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash')
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [14]:
llm.invoke("hi").content

'Hi there! How can I help you today?'

In [15]:
def data_ingestion(path):
    loader = WebBaseLoader(path)  
    data = loader.load()
    return data

In [16]:
from langchain.prompts import ChatPromptTemplate
def prompt_helper():
    template = """ Answer Based on the following context:
    {context}
    Question: {question}
    provide only helpful information.
    """
    prompt = ChatPromptTemplate.from_template(template)
    return prompt

In [17]:
def main():
    path = input("Enter the website link: ")
    data = data_ingestion(path)
    retriever = VectorStore(data,embedding)
    prompt = prompt_helper()
    chain = (
    {'context': retriever , 'question': RunnablePassthrough()}
    | prompt
    |llm
    |StrOutputParser()
    )
    question = input("Enter the question from the link: ")
    response = chain.invoke(question)
    print(response)

In [18]:
main()

The provided text does not contain any information about the color blue.
